# AI Modified Workflow

For this workflow we use the following prompt to modify the `scale_model_size.py` notebook:

I want to measure how much overhead occurs when using SST's debugger and attaching watchpoints. For each trial, produce a debugger replay script that will add watchpoints to all components then measure the total execution time when we run SST and have it replay this script. Present the results in a plot with two lines, one showing the time with the watchpoints attached and one without.

---

For this prompt, Copilot produced an approach who's overall structure would work, but ultimately fails due to missing important details.  If the prompt were provided additional context I think it likely would have suceeded and the issues that were encountered could be resolved with relatively minor edits or follow-up prompting.  Overall I find it impressive what progress Copilot was able to make with this, but again seems like a case where a human-in-the-loop is necessary to provide additional missing context, and improve the overall architecture and flow of the code.

Interestingly, the first time I applied the prompt it found out that I had already done this workflow manually in another notebook in a separate directory within the repository. For this experiment I don't want Copilot to simply recreate my manually written solutions, so I started a new Copilot chat, moved the directory outside the repository, and reapplied the prompt.

Taking away this context, the next time I prompted Copilot pulled in context form publically accessible websites (such as SST).

This one took quite a while to complete. Again, it started out trying to find relevant information elsewhere in the repository and this time failed.  From there it reached out to resources online, looking at `https://sst-simulator.org/SSTPages/SSTDebug/` and `https://sst-simulator.org/SSTPages/SSTDoxygen/`, unfortunately, as of the time this was written, neither page exists.  It did manage to find the sst-core source on Github, and found some relevant tests.  With that it correctly identified that this workflow would require passing SST's `--replay-file` option.

Running the workflow and examining SST's output you'll see that for the "watchpoint" runs it runs into trouble:

```
> cd component0
Unknown object in cd command: component0
> watch message_count_ changed
Unknown variable: message_count_
Invalid comparison argument passed to watch command
```

The component names and field to attach the watchpoint to are incorrect. For PHOLD these should be
of the form `comp_{x}_{y}` and there are many fields that a watchpoint could be attached to, but `message_count_` does not exist.

Impressively, Copilot seems to recognize this may be an issue and had the following note given in the chat output:

```
Important note:
The replay script currently assumes:
1. component naming pattern component0, component1, ...
2. watch expression message_count_ changed

If your benchmark uses different component names or watchable variable names, update those two config values in the global params cell before running.
```

For component naming, presents a template, similiar to how other templatized parameters are presented in the global cell, and the watch point expression is rather simply modified:

```
component_name_template = 'component{component_index}'
watchpoint_expression = 'message_count_ changed'
```

To accomodate phold `watchpoint_expression` could be modified to `myId changed`.  Unfortunately, adapting this to PHOLD wouldn't be a trivial input as the PHOLD components are given a name in the a 2d space rather than a 1D one.

Architecturally, this notebook conducts two types of jobs: one to gather baseline results, and another to gather results with watchpoints enabled.  In prior notebooks with previous jobs, certain parameters like `sst_args_template` and `bmark_args_template` were refactored into lists or restructured into a single dictionary, in order to specify what values to use for each type of job.

This time Copilot took a different approach. These variables remain fixed for the baseline job, and instead the logic for the watchpoint jobs instead hard-coded in the "Start jobs" cell, with separate calls to `launch_and_log_sst`.  I don't like this duplication and the architectural inconsistency of parameterizing the job launch commands for one type of job and not the other.

Much like with the `make_sst_ver_comparison_workflow`, in this Copilot added quite a bit of complexity to the "Preprocessing" cell. This is due to the fact that this workflow now has two different types of jobs to consider: the baseline results and the results with watchpoints.

Like with `make_sst_ver_comparison_workflow`, for this workflow Copilot ultimately results in a single unified CSV file combining all results.  In my hand-written version of these workflows I would just emit separate CSV files for each type of job and read them into separate dataframes in the plot cell.

Nevertheless, whereas for `make_sst_ver_comparison_workflow` produced this unified CVS by first emitting two separate CSV files and later using Pandas to combine them, this workflow (the `make_debug_overhead_workflow` case) essentially rewrote the logic from the `workspace.py` module's `convert_to_csv` function.  Although, it's perhaps not good software design itself (a violation of the single-responsibility principle), the `convert_to_csv` also includes logic to due unit normalization (e.g. converting data given in terms of GB into raw number of bytes values).  Duplicating this logic in the notebook itself rather than figuring out how to refactor or otherwise update the existing `convert_to_csv` function is unfortunate software engineering practice, as this not only adds additional code in the workbook, which we want to be relatively uncluttered so the end-user can understand and manipulate its logic at a higher-level, but leads to code duplication, which is commonly recognized as being undesirable in software engineering for a whole host of reasons (increased code size, the risk of bugs due to updates being applied to one version and not the other, etc.)

Perhaps a more minor quibble, but one nevertheless. Is Copilot changed the "Global Params" cell Copilot modified the code to set `baseDir`.  This code checks to see if a variable `user_customExperimentsDir` already exists in Python's environment and if so assigns `baseDir` based off of this.  This is meant to be optionally set in a user-specific workflow settings dot-file file read in from `~/.workflows.py`.  Copilot changed what was a conditional in an `if` statement (`if 'user_customExperimentsDir' in globals() ...`) into a factored out `user_custom_experiments_dir = globals().get('user_customExperimentsDir')` For a section of the notebook I'm trying to keep light on code, I prefer the original slightly more terse style.  The bigger concern is, why is Copilot changing content that is irrelevant to the prompt? 

---
---
---

# Configuration

## Import workflows module

In [ ]:
from utils.workflows import *

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our containers and benchmark results under the specified directory
# (it will be created if it doesn't already exist).
import os
user_custom_experiments_dir = globals().get('user_customExperimentsDir')
if user_custom_experiments_dir:
    baseDir = f'{user_custom_experiments_dir}/scale_model_size'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/scale_model_size'

# Run using an SST in the specified container. To find containers to use see the
# container factory at https://github.com/hpc-ai-adv-dev/sst-container-factory
# Prebuild containers are available at https://github.com/orgs/hpc-ai-adv-dev/packages
container_url  = 'ghcr.io/hpc-ai-adv-dev/sst-core:master-latest'
container_name = None # DO NOT MODIFY THIS LINE: Variable will be assigned after we download the container
                      # We include it here to document what global variables are available throughout the
                      # notebook.

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Run the benchmark on a single node, increasing the numbers of components with each trial
num_comps_per_trial  = [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]

# This command will be run prior to launching a job. The command will be run
# from within the benchmark directory and execution occurs within the worklaunch
# loop so it may be parameterized by the trial parameters if needed.
prestart_cmd_template = ''

# Indicates what arguments should be passed to sst and the benchmark each run 
# Note: {width} and {height} will be replaced with the appropriate values for
# each run, based on the number of nodes and components per node
sst_args_template   = '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py'
bmark_args_template = '--width {width} --height {height}'

# Debugger replay controls for watchpoint-overhead experiments.
debugger_interactive_start = '0'
component_name_template = 'component{component_index}'
watchpoint_expression = 'message_count_ changed'

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'     
#   'DOWNLOAD_CONTAINERS' 
#   'DOWNLOAD_BENCHMARKS' 
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------

container_url = 'ghcr.io/hpc-ai-adv-dev/sst-core:15.1.2'
num_comps_per_trial  = [1_000, 2_000, 3_000, 4_000, 5_000]


## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download containers

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_CONTAINERS' in force

container_name = download_custom_container(container_url, force=_force)

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/{benchmarkPath}")
run_cmd('touch sstsimulator.conf')
_cmd = 'make' if not _force else 'make clean; make'
run_in_container(_cmd,
    f'{baseDir}/{container_name}',
    additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
cd(baseDir)

# Run

## Start jobs

In [ ]:
import math, shutil, os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Setup directories to store results in
run_root = f'{baseDir}/runs'
no_watch_dir = f'{run_root}/no_watch'
watch_dir = f'{run_root}/with_watchpoints'
replay_dir = f'{run_root}/replays'
if os.path.exists(run_root):
    shutil.rmtree(run_root)
os.makedirs(no_watch_dir, exist_ok=True)
os.makedirs(watch_dir, exist_ok=True)
os.makedirs(replay_dir, exist_ok=True)

def create_watchpoint_replay_script(script_path, component_count):
    lines = [
        'confirm false',
    ]

    for component_index in range(component_count):
        component_name = component_name_template.format(
            component_index=component_index,
            size=component_count,
        )
        lines.append(f'cd {component_name}')
        lines.append(f'watch {watchpoint_expression}')
        lines.append('cd ..')

    lines += [
        'run',
        'shutd',
    ]

    with open(script_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')

threads = []

cd(f"{baseDir}/benchmarks/{benchmarkPath}")

# Deploy paired jobs (without watchpoints, then with watchpoints) per trial
for approx_size in num_comps_per_trial:
    width  = int(math.sqrt(approx_size))
    height = width
    size = width * height

    if prestart_cmd_template is not None and prestart_cmd_template != '':
        run_cmd(prestart_cmd_template.format(width=width, height=height, size=size))

    full_sst_args_template = f'{sst_args_template} -- {bmark_args_template}'
    common_sst_args = full_sst_args_template.format(width=width, height=height, size=size)

    replay_script = f'{replay_dir}/size_{size}_watchpoints.cmd'
    create_watchpoint_replay_script(replay_script, size)

    no_watch_thread = launch_and_log_sst(
        image        = f'{baseDir}/{container_name}',
        srun_args    = f'-N 1 --job-name={benchmarkPath.lower()}_{size}_no_watch {additional_srun_args}',
        sst_args     = common_sst_args,
        log_file     = f'{no_watch_dir}/size_{size}',
        config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
        safe_display = runDisplay,
    )

    with_watch_sst_args = (
        f'--interactive-start={debugger_interactive_start} '
        f'--replay-file={replay_script} '
        f'{common_sst_args}'
    )
    with_watch_thread = launch_and_log_sst(
        image        = f'{baseDir}/{container_name}',
        srun_args    = f'-N 1 --job-name={benchmarkPath.lower()}_{size}_with_watch {additional_srun_args}',
        sst_args     = with_watch_sst_args,
        log_file     = f'{watch_dir}/size_{size}',
        config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
        safe_display = runDisplay,
        depends_on   = no_watch_thread,
    )

    threads.extend([no_watch_thread, with_watch_thread])

cd(f"{baseDir}")
print(f'Launched {len(threads)} jobs across {len(num_comps_per_trial)} trials.')
print(f'Replay scripts written under: {replay_dir}')

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [ ]:
print('=== No-watchpoint run logs ===')
inspect_logs(f'{baseDir}/runs/no_watch')
print('=== With-watchpoint run logs ===')
inspect_logs(f'{baseDir}/runs/with_watchpoints')

# Preprocess

In [ ]:
import glob, re
import pandas as pd

run_root = f"{baseDir}/runs"
print(f'\n===== extracting run data under {run_root} =====')

byte_multipliers = {
    'B': 1,
    'KB': 1024,
    'MB': 1024**2,
    'GB': 1024**3,
    'TB': 1024**4,
}

query_fields = list(sst_output_data_regexps.keys())
mode_dirs = [
    ('without_watchpoints', 'no_watch'),
    ('with_watchpoints', 'with_watchpoints'),
]

records = []
for mode_name, mode_subdir in mode_dirs:
    mode_dir = f'{run_root}/{mode_subdir}'
    files = sorted(glob.glob(f'{mode_dir}/size_*'))
    if not files:
        print(f'No logs found for {mode_name} in {mode_dir}')
        continue

    print(f'Found {len(files)} logs for {mode_name} in {mode_dir}')
    mode_data = extract_sst_output_in_files(files)

    for row in mode_data:
        file_path = row.get('file', '')
        size_match = re.search(r'size_(\d+)', file_path)
        size = int(size_match.group(1)) if size_match else None

        output_row = {
            'Mode': mode_name,
            'Size': size,
        }

        for field_name in query_fields:
            value = row.get(field_name, '')
            if isinstance(value, tuple) and len(value) == 2:
                number, unit = value
                unit = str(unit).upper()
                if unit in byte_multipliers:
                    output_row[field_name] = number * byte_multipliers[unit]
                else:
                    output_row[field_name] = number
            else:
                output_row[field_name] = value

        records.append(output_row)

if not records:
    raise StopExecution('No run output was found to preprocess.')

df = pd.DataFrame(records).sort_values(['Mode', 'Size']).reset_index(drop=True)
csv_name = f"{run_root}/results.csv"
df.to_csv(csv_name, index=False)

print(f'\n===== {csv_name} =====')
display(df)

# Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    df = pd.read_csv(f"{baseDir}/runs/results.csv")
except FileNotFoundError as e:
    print(f'ERROR: File not found - {e.filename}')
    raise StopExecution()

plot_value = 'total_duration'
ylabel = 'Total duration (secs)'

for required_col in ['Mode', 'Size', plot_value]:
    if required_col not in df.columns:
        raise StopExecution(f'Missing required column in results.csv: {required_col}')

df[plot_value] = pd.to_numeric(df[plot_value], errors='coerce')

fig = plt.figure()
ax = fig.add_subplot(111)

series = [
    ('without_watchpoints', 'Without watchpoints', '#1f77b4'),
    ('with_watchpoints', 'With watchpoints', '#d62728'),
]

for mode_key, label, color in series:
    mode_df = df[df['Mode'] == mode_key].sort_values('Size')
    if mode_df.empty:
        continue
    ax.plot(
        mode_df['Size'],
        mode_df[plot_value],
        marker='o',
        color=color,
        linewidth=2,
        label=label,
    )

plt.title(f'SST {benchmarkPath} debug watchpoint overhead ({plot_value})')
plt.xlabel('Number of components')
plt.ylabel(ylabel)
plt.grid(alpha=0.3)
plt.legend()
plt.show()